# Google Colab Auto-AOI Generator

Upload two files from the app:

- the study video
- `colab-aoi-job-*.json`

This notebook samples the video, runs Florence-2 object detection, groups detections into simple tracks, and downloads an AOI JSON file that can be imported with **Import Colab AOIs** or **Load AOI JSON** in the app.

In [ ]:
!pip -q install transformers accelerate opencv-python pillow

import json
import math
import re
from pathlib import Path

import cv2
import torch
from google.colab import files
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
MODEL_ID = 'microsoft/Florence-2-base'

print('Using', DEVICE)

In [ ]:
uploaded = files.upload()
paths = [Path(name) for name in uploaded.keys()]
job_path = next(path for path in paths if path.suffix.lower() == '.json')
video_path = next(path for path in paths if path != job_path)

job = json.loads(job_path.read_text())
assert job.get('kind') == 'aoi-colab-job', 'Upload a colab-aoi-job JSON exported from the app.'

video_meta = job.get('video', {})
policy = job.get('aoiPolicy', {})
prompts = [prompt.lower() for prompt in policy.get('prompts', [])]
sample_interval = float(policy.get('sampleIntervalSec', 1.0))

print('Video:', video_path)
print('Projection:', video_meta.get('projection', 'equirectangular'))
print('Stereo:', video_meta.get('stereoLayout', 'mono'))
print('Prompts:', prompts)
print('Sample interval:', sample_interval)

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()

def normalize_yaw(value):
    normalized = ((value + 180) % 360) - 180
    return 180 if normalized == -180 and value > 0 else normalized

def slug(label):
    text = re.sub(r'[^a-z0-9]+', '-', label.lower()).strip('-')
    return text or 'generated-aoi'

def iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x1, y1 = max(ax1, bx1), max(ay1, by1)
    x2, y2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    denom = area_a + area_b - inter
    return inter / denom if denom else 0

def prompt_allowed(label):
    if not prompts:
        return True
    lowered = label.lower()
    return any(prompt in lowered or lowered in prompt for prompt in prompts)

def crop_stereo_eye(frame, stereo_layout):
    height, width = frame.shape[:2]
    if stereo_layout == 'side-by-side':
        return frame[:, :width // 2]
    if stereo_layout == 'top-bottom':
        return frame[:height // 2, :]
    return frame

def sample_frames(video_file, interval_sec, stereo_layout):
    cap = cv2.VideoCapture(str(video_file))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    source_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    source_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if frame_count else 0
    times = []
    t = 0.0
    while t <= max(duration, 0.01):
        times.append(round(t, 3))
        t += max(0.25, interval_sec)
    frames = []
    for t in times:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, frame = cap.read()
        if not ok:
            continue
        frame = crop_stereo_eye(frame, stereo_layout)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append((t, Image.fromarray(rgb)))
    cap.release()
    if stereo_layout == 'side-by-side':
        width, height = source_width // 2, source_height
    elif stereo_layout == 'top-bottom':
        width, height = source_width, source_height // 2
    else:
        width, height = source_width, source_height
    return frames, {
        'width': width,
        'height': height,
        'sourceWidth': source_width,
        'sourceHeight': source_height,
        'durationSec': duration,
        'eye': 'left',
    }

def detect_frame(image):
    task = '<OD>'
    inputs = processor(text=task, images=image, return_tensors='pt').to(DEVICE, DTYPE)
    with torch.inference_mode():
        generated_ids = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=1024,
            num_beams=3,
        )
    text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(text, task=task, image_size=image.size)
    result = parsed.get(task, {})
    labels = result.get('labels', [])
    boxes = result.get('bboxes', [])
    return [
        {'label': label, 'bbox': [float(v) for v in box]}
        for label, box in zip(labels, boxes)
        if prompt_allowed(label)
    ]

def box_to_keyframe(t, bbox, width, height, projection):
    x1, y1, x2, y2 = bbox
    x_min = max(0, min(1, x1 / width))
    x_max = max(0, min(1, x2 / width))
    y_min = max(0, min(1, y1 / height))
    y_max = max(0, min(1, y2 / height))
    if projection == 'flat':
        return {
            't': t,
            'xMin': round(min(x_min, x_max), 6),
            'xMax': round(max(x_min, x_max), 6),
            'yMin': round(min(y_min, y_max), 6),
            'yMax': round(max(y_min, y_max), 6),
        }
    return {
        't': t,
        'yawMin': round(normalize_yaw(min(x_min, x_max) * 360 - 180), 6),
        'yawMax': round(normalize_yaw(max(x_min, x_max) * 360 - 180), 6),
        'pitchMin': round(90 - max(y_min, y_max) * 180, 6),
        'pitchMax': round(90 - min(y_min, y_max) * 180, 6),
    }

def group_detections(detections, iou_threshold=0.25):
    tracks = []
    for detection in detections:
        best_track = None
        best_score = 0
        for track in tracks:
            if track['label'] != detection['label']:
                continue
            score = iou(track['last_bbox'], detection['bbox'])
            if score > best_score:
                best_score = score
                best_track = track
        if best_track and best_score >= iou_threshold:
            best_track['detections'].append(detection)
            best_track['last_bbox'] = detection['bbox']
        else:
            tracks.append({
                'label': detection['label'],
                'last_bbox': detection['bbox'],
                'detections': [detection],
            })
    return tracks

In [ ]:
stereo_layout = video_meta.get('stereoLayout', 'mono')
frames, video_stats = sample_frames(video_path, sample_interval, stereo_layout)
all_detections = []

for index, (t, image) in enumerate(frames, start=1):
    detections = detect_frame(image)
    for detection in detections:
        detection['t'] = t
    all_detections.extend(detections)
    print(f'{index}/{len(frames)} t={t:.2f}s detections={len(detections)}')

projection = video_meta.get('projection', 'equirectangular')
colors = ['#ffd166', '#5dd7c8', '#ff8a5c', '#8bd66f', '#ff4f9a', '#9fb7ff']
tracks = group_detections(all_detections)
aois = []
used_ids = set()

for idx, track in enumerate(tracks):
    if len(track['detections']) < 1:
        continue
    base_id = slug(track['label'])
    aoi_id = base_id
    suffix = 2
    while aoi_id in used_ids:
        aoi_id = f'{base_id}-{suffix}'
        suffix += 1
    used_ids.add(aoi_id)
    keyframes = [
        box_to_keyframe(
            det['t'],
            det['bbox'],
            video_stats['width'],
            video_stats['height'],
            projection,
        )
        for det in track['detections']
    ]
    first = keyframes[0]
    aois.append({
        'id': aoi_id,
        'label': track['label'],
        'color': colors[idx % len(colors)],
        'space': 'video' if projection == 'flat' else 'panorama',
        'generated': {
            'method': 'google-colab-florence2',
            'sampleIntervalSec': sample_interval,
            'projection': projection,
            'stereoLayout': stereo_layout,
            'eye': video_stats.get('eye', 'left'),
            'frameDetections': len(track['detections']),
        },
        **first,
        'keyframes': keyframes,
    })

output = {
    'video': {
        **video_meta,
        'width': video_stats['width'],
        'height': video_stats['height'],
        'sourceWidth': video_stats.get('sourceWidth'),
        'sourceHeight': video_stats.get('sourceHeight'),
        'eye': video_stats.get('eye', 'left'),
        'durationSec': round(video_stats['durationSec'], 3),
    },
    'aois': aois,
    'generatedBy': 'google-colab-florence2',
    'sourceJob': job,
}

out_path = Path('generated-colab-aois.json')
out_path.write_text(json.dumps(output, indent=2))
print(f'Generated {len(aois)} AOIs')
files.download(str(out_path))